# Initialise OpenBao

In [ ]:
import json
from pathlib import Path
import requests

LOCAL_CA_CERT = Path.cwd() / "certs" / "ca" / "ca.crt"

OPENBAO_URL = "https://openbao.localhost"
OPENBAO_INIT_FILE = Path().cwd() / "assets" / "openbao.json"

def initialise_openbao() -> dict:
    response = requests.post(
        f"{OPENBAO_URL}/v1/sys/init",
        json={
            "secret_shares": 1,
            "secret_threshold": 1,
        },
        verify=str(LOCAL_CA_CERT),
        timeout=30
    )

    response.raise_for_status()
    return response.json()

def get_openbao_init_from_file() -> dict | None:
    if not OPENBAO_INIT_FILE.exists():
        return None

    openbao_init = json.loads(OPENBAO_INIT_FILE.read_text(encoding="utf-8"))
    return openbao_init or None

def save_openbao_init(openbao_init: dict) -> None:
    OPENBAO_INIT_FILE.parent.mkdir(parents=True, exist_ok=True)
    OPENBAO_INIT_FILE.write_text(json.dumps(openbao_init, indent=2), encoding="utf-8")

def get_openbao_init():
    openbao_init = get_openbao_init_from_file()
    if openbao_init is not None:
        return openbao_init

    openbao_init = initialise_openbao()
    save_openbao_init(openbao_init)

    return openbao_init

In [ ]:
openbao_init = get_openbao_init()

# Unseal OpenBao

In [ ]:
unseal_key = openbao_init["keys"][0]

requests.post(
    f"{OPENBAO_URL}/v1/sys/unseal",
    json={"key": unseal_key},
    verify=str(LOCAL_CA_CERT),
    timeout=30
)

# Create OpenBao HTTPS client

In [ ]:
import requests

OPENBAO_URL = "https://openbao.localhost"
openbao_token = openbao_init["root_token"]

openbao = requests.Session()
openbao.headers.update({
    "X-Vault-Token": openbao_token
})
openbao.verify = str(LOCAL_CA_CERT)

In [ ]:
response = openbao.get(
    f"{OPENBAO_URL}/v1/auth/token/lookup-self",
    timeout=30,
)

response.raise_for_status()
token_info = response.json()
print(token_info)